# SG-019 — Encoding shard và dense index (driver notebook)

Notebook điều phối việc encode corpus thành shard có checksum + manifest và dựng **exact index**
qua module `scholargraph.retrieval.dense` (không viết lại logic trong notebook).

- Model pin (SG-017): `BAAI/bge-base-en-v1.5` @ revision `a5beb1e3e68b9ab74eb54cfd186867f64f240e1a`,
  CLS pooling, 768 dims, `max_length=512`, prefix **chỉ cho query**
  (`"Represent this sentence for searching relevant passages: "`), documents không prefix.
- Protocol `retrieval.dense`: cosine trên vector L2-normalized; `missing_embedding=fail_not_drop_document`.
- Shard: mặc định 1.000 docs/shard; mỗi shard có `.npy` + manifest (doc-ID đúng thứ tự, sha256,
  model/tokenizer revision, config hash). Resume: shard COMPLETED khớp checksum thì bỏ qua;
  config/model/revision lệch thì TỪ CHỐI (exit 1). Không dùng ANN; exact index có hash.
- Máy local chỉ có CPU nên `"gpu": "not_measured_local"`; mọi tốc độ/RAM là số đo thật của run.

**Lưu ý trung thực:** các cell encode chạy lại an toàn (resume); run thật của phiên được ghi trong
`management/logs/SESSION-20260917-041.md` và `artifacts/reports/encoding_profile.json`.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs" / "models" / "dense_encoder.json").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Project root with configs/models/dense_encoder.json not found; run from scholargraph/.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from scholargraph.retrieval.dense import load_dense_encoder_config

CONFIG = load_dense_encoder_config(
    PROJECT_ROOT / "configs" / "models" / "dense_encoder.json",
    PROJECT_ROOT / "configs" / "protocol_v1.json",
)
PIN = {
    "model_id": CONFIG.model_id,
    "revision": CONFIG.revision,
    "tokenizer_revision": CONFIG.revision,
    "pooling": CONFIG.pooling,
    "embedding_dimension": CONFIG.embedding_dimension,
    "max_length": CONFIG.max_length,
    "prefix_scope": CONFIG.prefix_scope,
    "prefix": CONFIG.prefix,
    "missing_embedding": CONFIG.missing_embedding,
    "similarity": CONFIG.similarity,
    "config_hash": CONFIG.config_hash,
    "namespace": CONFIG.namespace,
}
print(json.dumps(PIN, indent=2, ensure_ascii=False))

## Kho corpus cần encode

Thứ tự ưu tiên khi thời gian bị giới hạn: **SciFact → OpenAlex core → SCIDOCS**. Mỗi corpus được
đọc từ manifest dữ liệu thật (hash corpus được kiểm tra trước khi encode).

In [ ]:
CORPORA = [
    {"corpus_id": "scifact", "manifest": "data/manifests/beir_scifact.json", "count_key": ("counts", "corpus")},
    {"corpus_id": "openalex_core", "manifest": "data/manifests/openalex_core.json", "count_key": ("postprocess", "processed_works")},
    {"corpus_id": "scidocs", "manifest": "data/manifests/beir_scidocs.json", "count_key": ("counts", "corpus")},
]

inventory = {}
for item in CORPORA:
    manifest = json.loads((PROJECT_ROOT / item["manifest"]).read_text(encoding="utf-8"))
    node = manifest
    for key in item["count_key"]:
        node = node[key]
    inventory[item["corpus_id"]] = {"manifest": item["manifest"], "expected_documents": node}
print(json.dumps(inventory, indent=2, ensure_ascii=False))

## Ước lượng thời gian còn lại ("hai bạn cần hiểu")

Công thức COMPUTE_PLAN §5: thời gian ước tính = `docs_remaining / docs_per_second × 1.25` (25%
overhead tokenization/I/O/checkpoint). Chỉ dùng **throughput đo thật** của cùng model/device; chưa
đo thì báo chưa đo, không ngoại suy từ MiniLM và không bịa số.

In [ ]:
OVERHEAD_FACTOR = 1.25


def load_measured_throughput(corpus_id: str, namespace: str) -> float:
    """Read the measured docs_per_second of the latest encode run for a corpus."""

    summary_path = PROJECT_ROOT / "artifacts" / "indexes" / corpus_id / namespace / "encode_summary.json"
    if not summary_path.exists():
        raise ValueError(
            f"No measured throughput for {corpus_id}: {summary_path} is missing. "
            "Run at least one shard first; refusing to fabricate an estimate."
        )
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    docs_per_second = summary["timing"]["docs_per_second_this_run"]
    if not docs_per_second or docs_per_second <= 0:
        raise ValueError(f"Measured docs_per_second for {corpus_id} is {docs_per_second!r}; cannot estimate.")
    return float(docs_per_second)


def estimate_remaining(docs_remaining: int, docs_per_second: float, overhead: float = OVERHEAD_FACTOR) -> float:
    """Operational estimate in seconds: N / r * overhead (COMPUTE_PLAN §5)."""

    if docs_remaining < 0:
        raise ValueError(f"docs_remaining must be >= 0, got {docs_remaining}.")
    return docs_remaining / docs_per_second * overhead


def print_eta_table() -> None:
    total_seconds = 0.0
    measurable = True
    for item in CORPORA:
        corpus_id = item["corpus_id"]
        expected = inventory[corpus_id]["expected_documents"]
        try:
            rate = load_measured_throughput(corpus_id, CONFIG.namespace)
            seconds = estimate_remaining(expected, rate)
            total_seconds += seconds
            print(f"{corpus_id:>14}: {expected:>6} docs @ {rate:8.2f} docs/s -> ~{seconds / 60:6.1f} min (incl. {OVERHEAD_FACTOR:.2f}x overhead)")
        except ValueError as exc:
            measurable = False
            print(f"{corpus_id:>14}: {expected:>6} docs -> {exc}")
    if measurable and total_seconds > 0:
        print(f"{'TOTAL':>14}: ~{total_seconds / 60:6.1f} min theo throughput da do (cung model/device).")
    else:
        print("TOTAL: chua do du throughput -> chua uoc tinh duoc, khong suy dien tu model khac.")


print_eta_table()

## Encode từng corpus (resume-safe, exit code bắt buộc 0)

Mỗi lệnh gọi CLI dừng ngay khi exit code != 0 (fail-loud theo ADR-013). Batch size CPU mặc định
16; giảm 8 → 4 nếu RAM khan hiếm.

In [ ]:
def run_dense_cli(args: list[str]) -> str:
    """Run one dense CLI command; raise on any non-zero exit (no silent bypass)."""

    command = [sys.executable, "-m", "scholargraph.retrieval.dense", *args]
    print("+", " ".join(command), flush=True)
    started = time.perf_counter()
    env = dict(os.environ)
    env["PYTHONPATH"] = str(SRC)
    result = subprocess.run(command, cwd=str(PROJECT_ROOT), capture_output=True, text=True, env=env)
    elapsed = time.perf_counter() - started
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:], file=sys.stderr)
        raise RuntimeError(f"CLI failed with exit code {result.returncode} after {elapsed:.1f}s: {' '.join(args)}")
    print(f"[exit 0 in {elapsed:.1f}s]")
    return result.stdout


for item in CORPORA:
    run_dense_cli([
        "encode",
        "--corpus", item["corpus_id"],
        "--shard-size", "1000",
        "--batch-size", "16",
    ])

## Exact index cho từng corpus (không ANN)

In [ ]:
for item in CORPORA:
    run_dense_cli(["index", "--corpus", item["corpus_id"]])
    run_dense_cli(["verify", "--corpus", item["corpus_id"]])

## Ghi `artifacts/reports/encoding_profile.json`

Profile tổng hợp biến đếm thật: docs/shards/checksum đã xác minh, throughput đo được từng corpus,
RAM peak, revision pin (model + tokenizer), config hash, trạng thái resume, và
`gpu: not_measured_local` (máy này không có CUDA; không bịa số GPU).

In [ ]:
namespace_args = []
for item in CORPORA:
    namespace_args += ["--corpus-namespace", f"{item['corpus_id']}={CONFIG.namespace}"]

run_dense_cli(["profile", "--output", "artifacts/reports/encoding_profile.json", *namespace_args])

profile = json.loads((PROJECT_ROOT / "artifacts" / "reports" / "encoding_profile.json").read_text(encoding="utf-8"))
print(json.dumps(profile["totals"], indent=2))
print(json.dumps({cid: block["status"] for cid, block in profile["corpora"].items()}, indent=2))

## Cách diễn giải (bắt buộc đọc)

1. **Vector thuộc paper nào:** thứ tự dòng của mọi `.npy` bằng đúng `doc_ids` trong manifest shard;
   exact index nối shard theo đúng thứ tự đó và `doc_ids.json` cho phép tra ngược mọi dòng về doc
   ID canonical (`beir:scifact:<raw>`, `openalex:W...`, `beir:scidocs:<raw>`).
2. **Thời gian còn lại:** cell ETA nhân throughput đã đo của run này với số docs còn lại và hệ số
   overhead 1.25; chưa đo thì từ chối ước lượng (không ngoại suy từ MiniLM theo COMPUTE_PLAN §5).
3. **GPU:** trường `gpu` là `not_measured_local` — máy không có CUDA; mọi số là của CPU.
4. **Truncation:** `truncation.documents_over_limit / documents_measured` là tỷ lệ cắt thật ở
   `max_length=512` (yêu cầu MODEL_CARD §6.3).
5. Đây là compute/manifest artifact; không phải benchmark retrieval-quality (SG-026 mới đo điểm).